# Pinecone RAG Training Pipeline

This notebook provides a thorough integration test of the Pinecone corpus backend. It walks through the full 4-step pipeline:

1. **Point to dataset** - Chunk documents and upload to a Pinecone index with vector embeddings
2. **Create synthetic QA** - Generate question-answer pairs with CgftPipeline
3. **Create env** - Configure the Pinecone search environment
4. **Run training job** - Launch the training

Pinecone is **vector-only** (no BM25/lexical search), so an embedding function is **required**.

## Setup

Install dependencies and configure API credentials.

In [2]:
# For Google Colab - clone repo and install
# !git clone https://github.com/cgftinc/synthetic-data-prep-lib.git
# %cd synthetic-data-prep-lib
# %pip install -e ".[pinecone]" openai
%pip install -e /Users/thariqridha/Projects/cgft_projects/cgft

/Users/thariqridha/.venvs/cgft-pinecone/bin/python: No module named pip
Note: you may need to restart the kernel to use updated packages.


In [3]:
# Local development setup
%load_ext autoreload
%autoreload 2

import sys
from pathlib import Path

cwd = Path.cwd()
candidate_roots = [cwd, cwd.parent]
repo_root = next((p for p in candidate_roots if (p / "src" / "cgft").exists()), cwd)
src_path = repo_root / "src"
if src_path.exists() and str(src_path) not in sys.path:
    sys.path.insert(0, str(src_path))

In [4]:
import cgft
from cgft.utils import ensure_safe_python_version

ensure_safe_python_version()

In [5]:
# Configuration

# Pinecone credentials — get your API key at https://app.pinecone.io/
PINECONE_API_KEY = ""
INDEX_NAME = "cgft-test"           # Name of your Pinecone index
INDEX_HOST = ""           # Optional: direct host URL (e.g. "https://my-index-abc123.svc.aped-1234.pinecone.io")
NAMESPACE = ""            # Optional: namespace within the index (default "")

# Pinecone embedding model — uses Pinecone's built-in Inference API.
# No external embedding credentials needed.
# Available models: "multilingual-e5-large" (1024d), "llama-text-embed-v2" (1024d)
EMBED_MODEL = "multilingual-e5-large"

# Optional: custom embedding function for external providers (e.g. Azure OpenAI).
# Set to None to use Pinecone's built-in embeddings above.
# If you provide embed_fn, EMBED_MODEL is ignored and the index dimensions must
# match your function's output.
EMBED_FN = None

# To use Azure OpenAI embeddings instead, uncomment and fill in:
# from openai import AzureOpenAI
# _openai = AzureOpenAI(api_key="...", api_version="...", azure_endpoint="...")
# EMBED_FN = lambda texts: [
#     item.embedding for item in sorted(
#         _openai.embeddings.create(model="text-embedding-3-small", input=texts).data,
#         key=lambda x: x.index,
#     )
# ]

# CGFT API key — create one at https://app.cgft.io/account/api-keys
API_KEY = ""
BASE_URL = "https://app.cgft.io"

# LLM API credentials — used by CgftPipeline for QA generation, filtering, and refinement
LLM_API_KEY = ""
LLM_BASE_URL = "https://judge-elsa.services.ai.azure.com/openai/v1/"

# Judge LLM credentials — used at reward time to evaluate correctness
JUDGE_API_KEY = LLM_API_KEY
JUDGE_BASE_URL = LLM_BASE_URL
JUDGE_MODEL = "grok-4-fast-reasoning"

# Dataset configuration
DOCS_PATH = "./samples/posthog/docs/"

# Corpus intent used for corpus profiling (summary + example queries)
CORPUS_DESCRIPTION = "Posthog documentation including docs, company policy, etc."
EXAMPLE_QUERIES = [
    "how to feature flag",
    "setup reverse proxy to avoid ad blockers",
    "posthog install nextjs",
]

# QA generation configuration (CgftPipeline)
TOTAL_SAMPLES = 30
OUTPUT_DIR = "outputs/pinecone_rag"

# Optional: name for your training experiment
EXPERIMENT_NAME = None  # e.g. "posthog-pinecone-v1"
EXPERIMENT_PREFIX = None  # e.g. "posthog-pinecone"

# Optional: custom field mapping for "bring your own index" scenarios.
FIELD_MAPPING = None  # e.g. {"description": "content", "path": "file_path"}

## Step 0: Build Embedding Function

If `EMBED_FN` is `None`, Pinecone's built-in Inference API handles embedding automatically — no setup needed. This cell just runs a quick smoke test to verify.

In [6]:
if EMBED_FN is not None:
    embed_fn = EMBED_FN
    test_vec = embed_fn(["hello world"])
    print(f"Custom embed_fn: dimension={len(test_vec[0])}")
else:
    # Smoke-test Pinecone's built-in Inference API
    from pinecone import Pinecone as _Pc
    _pc = _Pc(api_key=PINECONE_API_KEY)
    _result = _pc.inference.embed(
        model=EMBED_MODEL,
        inputs=["hello world"],
        parameters={"input_type": "query"},
    )
    print(f"Pinecone Inference ({EMBED_MODEL}): dimension={len(_result.data[0].values)}")
    print(f"First 5 values: {_result.data[0].values[:5]}")
    embed_fn = None  # source/env will build it internally

Pinecone Inference (multilingual-e5-large): dimension=1024
First 5 values: [0.020111083984375, -0.0167694091796875, 0.00646209716796875, -0.04345703125, 0.044342041015625]


## Step 1: Point to Dataset

Chunk your documents and upload to a Pinecone index in one line.

> **Bring your own index?** If you already have a populated Pinecone index, skip `populate_from_folder` and set `FIELD_MAPPING` above to map your metadata fields to ours.

> **Note:** Make sure your Pinecone index has the correct dimensions for your embedding model (e.g. 1536 for `text-embedding-3-small`).

In [7]:
from pinecone import Pinecone, ServerlessSpec

pc = Pinecone(api_key=PINECONE_API_KEY)

# Probe embedding dimension from the model we'll use.
if embed_fn is not None:
    EMBED_DIMENSION = len(embed_fn(["dim probe"])[0])
else:
    _probe = pc.inference.embed(model=EMBED_MODEL, inputs=["dim probe"], parameters={"input_type": "query"})
    EMBED_DIMENSION = len(_probe.data[0].values)
print(f"Embedding dimension: {EMBED_DIMENSION}")

existing = [idx.name for idx in pc.list_indexes()]
if INDEX_NAME in existing:
    desc = pc.describe_index(INDEX_NAME)
    if desc.dimension != EMBED_DIMENSION:
        print(
            f"⚠ Index '{INDEX_NAME}' has dimension {desc.dimension} but embed model "
            f"'{EMBED_MODEL}' produces {EMBED_DIMENSION}."
        )
        confirm = input(f"Delete index '{INDEX_NAME}' and recreate with {EMBED_DIMENSION} dims? [y/N]: ")
        if confirm.strip().lower() == "y":
            pc.delete_index(INDEX_NAME)
            existing.remove(INDEX_NAME)
        else:
            raise ValueError("Dimension mismatch. Change EMBED_MODEL or delete the index manually.")
    else:
        print(f"Index '{INDEX_NAME}' already exists with correct dimension ({EMBED_DIMENSION})")

if INDEX_NAME not in existing:
    print(f"Creating index '{INDEX_NAME}' (dim={EMBED_DIMENSION}, metric=cosine)...")
    pc.create_index(
        name=INDEX_NAME,
        dimension=EMBED_DIMENSION,
        metric="cosine",
        spec=ServerlessSpec(cloud="aws", region="us-east-1"),
    )
    print("Index created. Waiting for it to be ready...")
    import time
    while not pc.describe_index(INDEX_NAME).status.get("ready"):
        time.sleep(2)
    print("Index ready!")

from cgft.corpus.pinecone.source import PineconeChunkSource
from cgft.chunkers.markdown import MarkdownChunker

source = PineconeChunkSource(
    api_key=PINECONE_API_KEY,
    index_name=INDEX_NAME,
    index_host=INDEX_HOST or None,
    namespace=NAMESPACE,
    embed_fn=embed_fn,            # None → uses Pinecone Inference API
    embed_model=EMBED_MODEL,
    field_mapping=FIELD_MAPPING,
)

# Use a small subset for quick validation — chunk then trim to ~50 chunks
chunker = MarkdownChunker(min_char=1024, max_char=2048, chunk_overlap=128)
collection = chunker.chunk_folder(DOCS_PATH, file_extensions=[".md", ".mdx"])
small_collection = collection[:50]  # first 50 chunks only
print(f"Using {len(small_collection)} of {len(collection)} chunks for validation")

source.populate_from_chunks(small_collection)

Embedding dimension: 1024
Index 'cgft-test' already exists with correct dimension (1024)


/Users/thariqridha/.venvs/cgft-pinecone/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Using 0 of 0 chunks for validation

Uploading 0 chunks to Pinecone index 'cgft-test'...


Uploading batches: 0it [00:00, ?it/s]


Upload complete! 0 chunks written to index.


### Step 1a: Verify the corpus

Quick sanity checks: sample chunks, search, capabilities, and file-structure awareness.

In [8]:
from pprint import pprint

# --- Capabilities ---
caps = source.get_search_capabilities()
print("=== Search Capabilities ===")
pprint(caps)
print(f"\nBackend: {caps['backend']}")
print(f"Modes: {caps['modes']}")
assert "vector" in caps["modes"], "Expected vector mode"
assert "lexical" not in caps["modes"], "Pinecone should NOT have lexical mode"
print("Capabilities check PASSED")

# --- Sample chunks ---
print("\n=== Sample Chunks ===")
samples = source.sample_chunks(n=3, min_chars=100)
print(f"Sampled {len(samples)} chunks (min 100 chars)")
for i, chunk in enumerate(samples):
    print(f"\n--- Chunk {i+1} ({len(chunk.content)} chars) ---")
    print(f"  file_path: {chunk.get_metadata('file_path')}")
    print(f"  chunk_index: {chunk.get_metadata('chunk_index')}")
    print(f"  content preview: {chunk.content[:120]}...")
assert len(samples) == 3, f"Expected 3 samples, got {len(samples)}"
print("\nSample check PASSED")

=== Search Capabilities ===
{'backend': 'pinecone',
 'constraints': {'max_top_k': 10000, 'vector_dimensions': None},
 'filter_ops': {'field': {'lte', 'gte', 'eq', 'in'},
                'logical': {'and', 'or', 'not'}},
 'graph_expansion': False,
 'modes': {'vector'},
 'ranking': {'cosine'}}

Backend: pinecone
Modes: {'vector'}
Capabilities check PASSED

=== Sample Chunks ===
Sampled 3 chunks (min 100 chars)

--- Chunk 1 (358 chars) ---
  file_path: self-host/index.mdx
  chunk_index: 5
  content preview: class diehard,privacy,worried,better_cloud,risk,support,try_it decision
class cloud_free,cloud_cheaper endpoint
```  
Yo...

--- Chunk 2 (837 chars) ---
  file_path: advanced/proxy.mdx
  chunk_index: 2
  content preview: ### Does PostHog provide static IP addresses?  
No. Our domains use AWS infrastructure with load balancing, which rotate...

--- Chunk 3 (1009 chars) ---
  file_path: cdp/destinations/intercom.md
  chunk_index: 1
  content preview: ## FAQ  
### Can I send session repla

In [9]:
from cgft.corpus.search_schema.search_types import SearchSpec

# --- Vector search via search() ---
print("=== Vector Search (SearchSpec) ===")
query_vec = source.embed_query("how to set up feature flags")
results = source.search(SearchSpec(
    mode="vector",
    vector_query=query_vec,
    top_k=3,
))
print(f"Found {len(results)} results for 'how to set up feature flags'")
for i, chunk in enumerate(results):
    print(f"\n  Result {i+1} ({len(chunk.content)} chars):")
    print(f"    {chunk.content[:150]}...")
assert len(results) > 0, "Expected at least 1 search result"
print("\nVector search check PASSED")

# --- search_text() convenience method ---
print("\n=== search_text() ===")
text_results = source.search_text("reverse proxy setup", top_k=3)
print(f"Found {len(text_results)} results for 'reverse proxy setup'")
assert len(text_results) > 0, "Expected at least 1 result from search_text"
print("search_text check PASSED")

# --- search_content() for cloudpickle safety ---
print("\n=== search_content() ===")
content_results = source.search_content(SearchSpec(
    mode="vector",
    vector_query=query_vec,
    top_k=3,
))
print(f"Got {len(content_results)} content strings")
assert all(isinstance(c, str) for c in content_results), "Expected strings"
print("search_content check PASSED")

# --- Verify lexical mode is rejected ---
print("\n=== Lexical mode rejection ===")
try:
    source.search(SearchSpec(mode="lexical", text_query="test", top_k=5))
    assert False, "Should have raised UnsupportedSearchModeError"
except Exception as e:
    print(f"Correctly rejected lexical mode: {type(e).__name__}")
    print(f"  {e}")
print("Lexical rejection check PASSED")

=== Vector Search (SearchSpec) ===
Found 3 results for 'how to set up feature flags'

  Result 1 (1058 chars):
    ## Gradual Rollouts  
There are many occasions when you might want to roll out a feature to your users slowly. Maybe you only want to enable it for Be...

  Result 2 (338 chars):
    ## Further reading  
- [Targeting feature flags on groups, pages, machines, and more](/docs/feature-flags/targeting-groups)
- [How to bootstrap featur...

  Result 3 (1100 chars):
    ### Tracking feature flag rollouts  
**Scenario**: A feature rollout caused issues and you need to identify exactly when and how the flag was changed....

Vector search check PASSED

=== search_text() ===
Found 3 results for 'reverse proxy setup'
search_text check PASSED

=== search_content() ===
Got 3 content strings
search_content check PASSED

=== Lexical mode rejection ===
Correctly rejected lexical mode: UnsupportedSearchModeError
  [pinecone] unsupported search mode 'lexical'. Supported modes: ['vector']
Le

In [10]:
# --- File-structure awareness ---
print("=== File-Structure Awareness ===")
file_aware = source._files.check()
print(f"File-aware: {file_aware}")

if file_aware:
    all_paths = source._files.get_all_file_paths()
    print(f"Found {len(all_paths)} unique file paths")
    for p in all_paths[:5]:
        print(f"  {p}")
    if len(all_paths) > 5:
        print(f"  ... and {len(all_paths) - 5} more")

    top_level = source.get_top_level_chunks()
    print(f"\nTop-level chunks: {len(top_level)}")
    assert len(top_level) > 0, "Expected top-level chunks"
    print("Top-level check PASSED")

    # Chunk context (prev/next neighbor)
    test_chunk = samples[0]
    ctx = source.get_chunk_with_context(test_chunk)
    print(f"\n=== Chunk Context ===")
    print(f"  chunk_content length: {len(ctx['chunk_content'])}")
    print(f"  prev_chunk_preview: {ctx['prev_chunk_preview'][:80]}...")
    print(f"  next_chunk_preview: {ctx['next_chunk_preview'][:80]}...")
    print("Context check PASSED")
else:
    print("Skipping file-structure tests (index lacks file_path/chunk_index)")

# --- search_related ---
print("\n=== search_related ===")
primary = samples[0]
related = source.search_related(primary, ["feature flags", "analytics"], top_k=3)
print(f"Found {len(related)} related chunks")
for r in related:
    print(f"  queries={r['queries']}, same_file={r['same_file']}, score={r['max_score']:.3f}")
    print(f"    content: {r['chunk'].content[:100]}...")
print("search_related check PASSED")

=== File-Structure Awareness ===
File-aware: False
Skipping file-structure tests (index lacks file_path/chunk_index)

=== search_related ===
Found 6 related chunks
  queries=['feature flags'], same_file=False, score=0.856
    content: ---
title: Feature Flag Use Cases
sidebar: Docs
showTitle: true
---  
Feature flags are a very power...
  queries=['feature flags'], same_file=False, score=0.847
    content: ## Gradual Rollouts  
There are many occasions when you might want to roll out a feature to your use...
  queries=['feature flags'], same_file=False, score=0.843
    content: ## Kill Switches  
You don't need feature flags _per se_ to implement kill switches, but having the ...
  queries=['analytics'], same_file=False, score=0.838
    content: if ( ! function_exists( 'add_posthog' ) ) :
function add_posthog() {
?>
<script>
!function(t,e){var ...
  queries=['analytics'], same_file=False, score=0.837
    content: ```php
if ( ! function_exists( 'add_posthog' ) ) :
function add_posthog()

### Step 1b: Verify filter mapper

Test that our FilterPredicate AST correctly translates to Pinecone filter dicts.

In [11]:
from cgft.corpus.search_schema.search_types import FieldPredicate, AndPredicate, SearchSpec

# --- Filtered vector search ---
print("=== Filtered Search ===")

# Search with a metadata filter (file_path equals a known path)
if file_aware and all_paths:
    target_path = all_paths[0]
    print(f"Filtering to file_path = '{target_path}'")

    filter_pred = FieldPredicate(field="file_path", op="eq", value=target_path)
    filtered_results = source.search_text("documentation", top_k=5, filter=filter_pred)
    print(f"Found {len(filtered_results)} results from '{target_path}'")
    for chunk in filtered_results:
        fp = chunk.get_metadata("file_path")
        print(f"  file_path={fp}, content={chunk.content[:80]}...")
        assert fp == target_path, f"Filter failed: expected '{target_path}', got '{fp}'"
    print("Filtered search check PASSED")

    # Test AND filter
    print("\n--- AND filter ---")
    and_filter = AndPredicate(clauses=(
        FieldPredicate(field="file_path", op="eq", value=target_path),
        FieldPredicate(field="char_count", op="gte", value=100),
    ))
    and_results = source.search_text("setup", top_k=3, filter=and_filter)
    print(f"AND filter returned {len(and_results)} results")
    print("AND filter check PASSED")
else:
    print("Skipping filter tests (no file-aware metadata)")

print("\n=== All Step 1 checks PASSED ===")

=== Filtered Search ===
Skipping filter tests (no file-aware metadata)

=== All Step 1 checks PASSED ===


### Step 1c: Verify mode-aware linkers

Test that the linkers and anchor_utils correctly detect Pinecone as a vector-only backend and generate appropriate search queries.

In [12]:
from cgft.qa_generation.anchor_utils import (
    _best_search_mode,
    generate_search_queries,
    generate_vector_queries,
)

# --- Mode detection ---
print("=== Mode-Aware Query Generation ===")
mode = _best_search_mode(source)
print(f"Detected best mode for Pinecone source: '{mode}'")
assert mode == "vector", f"Expected 'vector', got '{mode}'"
print("Mode detection PASSED")

# --- Query generation ---
test_chunk = samples[0]
print(f"\nTest chunk: {test_chunk.content[:100]}...")

# generate_search_queries should dispatch to vector strategy
queries = generate_search_queries(test_chunk, 3, source=source)
print(f"\ngenerate_search_queries (mode-aware, n=3):")
for q in queries:
    print(f"  '{q[:100]}'")
assert len(queries) > 0, "Expected at least 1 query"
print("Mode-aware query generation PASSED")

# Direct vector query generation
vec_queries = generate_vector_queries(test_chunk, 3)
print(f"\ngenerate_vector_queries (direct, n=3):")
for q in vec_queries:
    print(f"  '{q[:100]}'")
print("Vector query generation PASSED")

# --- Linker auto-prompt selection ---
from cgft.qa_generation.linkers import (
    AdaptiveChunkLinker,
    RELATED_CHUNK_SYSTEM_PROMPT,
    RELATED_CHUNK_VECTOR_SYSTEM_PROMPT,
)

# When we build a default linker with a Pinecone source, it should pick the vector prompt
# (We pass client=None so only StructuralChunkLinker is used, but the prompt selection still runs)
adaptive = AdaptiveChunkLinker.default(source=source, client=None)
print("\nAdaptiveChunkLinker.default() created successfully with Pinecone source")
print("Linker integration check PASSED")

ImportError: cannot import name '_best_search_mode' from 'cgft.qa_generation.anchor_utils' (/Users/thariqridha/Projects/cgft_projects/cgft/.claude/worktrees/pinecone/src/cgft/qa_generation/anchor_utils.py)

## Step 2: Create Synthetic QA

Generate synthetic QA pairs with `CgftPipeline` using the Pinecone corpus from Step 1.

In [ ]:
from cgft.qa_generation.cgft_models import (
    CgftPipelineConfig,
    CorpusConfig,
    CorpusContextConfig,
    FilteringConfig,
    GenerationConfig,
    GroundingLLMFilterConfig,
    LLMDirectGenerationConfig,
    OutputConfig,
    PlatformConfig,
    RetrievalLLMFilterConfig,
    TargetsConfig,
)
from cgft.qa_generation.cgft_pipeline import CgftPipeline

LLM_MODEL = "grok-4-fast-reasoning"

cfg = CgftPipelineConfig(
    platform=PlatformConfig(api_key=API_KEY, base_url=BASE_URL, llm_api_key=LLM_API_KEY, llm_base_url=LLM_BASE_URL),
    corpus=CorpusConfig(corpus_name="test", min_chunk_chars=400),
    corpus_context=CorpusContextConfig(
        description=CORPUS_DESCRIPTION,
        example_queries=EXAMPLE_QUERIES,
        num_top_level_samples=4,
        num_random_samples=4,
        min_chunk_chars=400,
    ),
    generation=GenerationConfig(
        llm_direct=LLMDirectGenerationConfig(model=LLM_MODEL),
    ),
    filtering=FilteringConfig(
        retrieval_llm=RetrievalLLMFilterConfig(judge_model=LLM_MODEL),
        grounding_llm=GroundingLLMFilterConfig(judge_model=LLM_MODEL),
    ),
    targets=TargetsConfig(
        total_samples=TOTAL_SAMPLES,
        qa_type_distribution={
            "lookup": 1,
            "co_located_multi_hop": 0,
            "cross_document_multi_hop": 0,
            "sequential_reasoning": 0,
            "synthesis": 0.0,
        },
    ),
    output=OutputConfig(dir=OUTPUT_DIR, train_jsonl="train.jsonl", eval_jsonl="eval.jsonl"),
)

cfg.resolve_api_keys()
print("generator model:", cfg.generation.llm_direct.model)
print("generator base_url:", cfg.generation.llm_direct.base_url)
print("retrieval judge model:", cfg.filtering.retrieval_llm.judge_model)
print("grounding judge model:", cfg.filtering.grounding_llm.judge_model)

generator model: grok-4-fast-reasoning
generator base_url: https://judge-elsa.services.ai.azure.com/openai/v1/
retrieval judge model: grok-4-fast-reasoning
grounding judge model: grok-4-fast-reasoning


In [ ]:
# Reuse the already-loaded corpus source from Step 1.
import importlib

import cgft.qa_generation.cgft_pipeline as _cgft_pipeline_mod
import cgft.qa_generation.generators.direct_llm as _direct_llm_mod

# Force-refresh generation modules in case notebook state still has stale classes.
importlib.reload(_direct_llm_mod)
importlib.reload(_cgft_pipeline_mod)
CgftPipeline = _cgft_pipeline_mod.CgftPipeline

cgft_pipeline = CgftPipeline(cfg, source_factory=lambda _cfg: source)
result = cgft_pipeline.run()

train_data = result["train_dataset"]
eval_data = result["eval_dataset"]
stats = result["stats"]

[1/8] Loading chunks from corpus...
[1/8] Loaded 30 seed chunks from corpus
[2/8] Building corpus profile...
[2/8] Built corpus profile
[3/8] Generating entity patterns...
[3/8] Generated entity patterns: 9 entities, 14 patterns
[4/8] Created 30 generation tasks
[5/8] Generating 30 QA candidates...


Processing prompts: 100%|██████████| 30/30 [00:25<00:00,  1.19it/s]
Skipping task task_00010: failed to parse QA response.
Skipping task task_00026: failed to parse QA response.
Skipping task task_00022: failed to parse QA response.
Skipping task task_00006: failed to parse QA response.
Skipping task task_00005: failed to parse QA response.
Skipping task task_00012: failed to parse QA response.
Skipping task task_00011: failed to parse QA response.
Skipping task task_00015: failed to parse QA response.
Skipping task task_00009: failed to parse QA response.
Skipping task task_00025: failed to parse QA response.
Skipping task task_00029: failed to parse QA response.
Skipping task task_00021: failed to parse QA response.
Skipping task task_00016: failed to parse QA response.
Skipping task task_00001: failed to parse QA response.
Skipping task task_00013: failed to parse QA response.
Skipping task task_00018: failed to parse QA response.
Skipping task task_00002: failed to parse QA respons

[5/8] Generated 4 QA candidates (26 failed to parse)
[6/8] Starting filtering (max 4 rounds)...
  Round 1/5: 4 items to evaluate
    deterministic_guards: 4 passed, 0 need refinement, 0 rejected
    grounding_llm: evaluating 0 items...


Processing prompts: 100%|██████████| 4/4 [00:19<00:00,  4.99s/it]


    grounding_llm: 4 passed, 0 need refinement, 0 rejected
    retrieval_too_easy_llm: evaluating 0 items...


Processing prompts: 100%|██████████| 4/4 [00:07<00:00,  1.81s/it]


    retrieval_too_easy_llm: 4 passed, 0 need refinement, 0 rejected
[7/8] Transformed 4 passed items
[8/8] Formatted output: 3 train, 1 eval

Pipeline complete: 4 passed, 0 rejected (0 regenerations)


### Step 2a: Verify Search Environment

Test the RL environment independently before launching training.

Uses `SearchEnv` with `PineconeSearch` — the new unified pattern.
The legacy `SearchEnv` also works (it's a thin wrapper).

In [ ]:
import asyncio
import nest_asyncio
nest_asyncio.apply()

from cgft.corpus.pinecone.search import PineconeSearch
from cgft.envs.search_env import SearchEnv

# Option 1 (recommended): SearchEnv + PineconeSearch
search = PineconeSearch(
    api_key=PINECONE_API_KEY,
    index_name=INDEX_NAME,
    embed_fn=embed_fn,            # None → uses Pinecone Inference API
    embed_model=EMBED_MODEL,
    index_host=INDEX_HOST or None,
    namespace=NAMESPACE,
    field_mapping=FIELD_MAPPING,
)
env = SearchEnv(
    search=search,
    judge_base_url=JUDGE_BASE_URL,
    judge_api_key=JUDGE_API_KEY,
    judge_model=JUDGE_MODEL,
)

# Option 2 (legacy, still works):
# from cgft.envs.search_env import SearchEnv
# env = SearchEnv(
#     pinecone_api_key=PINECONE_API_KEY, index_name=INDEX_NAME,
#     embed_fn=embed_fn, embed_model=EMBED_MODEL,
#     judge_base_url=JUDGE_BASE_URL, judge_api_key=JUDGE_API_KEY,
#     judge_model=JUDGE_MODEL,
# )

# --- List tools ---
tools = asyncio.run(env.list_tools())
print("=== Search Environment Tools ===")
for tool in tools:
    print(f"  {tool.name}: {tool.description}")
    print(f"    schema: {tool.input_schema}")
assert len(tools) == 1
assert tools[0].name == "search"
print("Tool listing check PASSED")

# --- Run search tool ---
print("\n=== Search Tool Execution ===")
search_result = asyncio.run(env._search_tool(query="how to install posthog"))
print(search_result[:500])
assert not search_result.startswith("Error")
print("\nSearch tool execution PASSED")

# --- Empty query ---
print("\n=== Empty Query Handling ===")
empty_result = asyncio.run(env._search_tool(query=""))
print(f"Empty query result: {empty_result}")
assert empty_result.startswith("Error")
print("Empty query check PASSED")

# --- Dataset preprocessing ---
print("\n=== Dataset Preprocessing ===")
example = {"question": "What is a feature flag?", "answer": "A feature flag is..."}
std = SearchEnv.dataset_preprocess(example)
print(f"result: {std}")
assert std["prompt"] == "What is a feature flag?"
assert std["ground_truth"] == "A feature flag is..."
print("Preprocessing check PASSED")

print("\n=== All environment checks PASSED ===")

## Step 3: Launch Training

Upload datasets and environment, then launch the training job.

In [ ]:
import cgft
from cgft.corpus.pinecone.search import PineconeSearch
from cgft.envs.search_env import SearchEnv
from cgft.trainer.pipeline import train

# Build the search client (pickle-safe — survives remote training)
search = PineconeSearch(
    api_key=PINECONE_API_KEY,
    index_name=INDEX_NAME,
    index_host=INDEX_HOST or None,
    namespace=NAMESPACE,
    embed_model=EMBED_MODEL,
    field_mapping=FIELD_MAPPING,
)

# Set to True when your Pinecone index is reachable from CGFT's cloud.
# This is usually true since Pinecone is a hosted service.
VALIDATE_REMOTELY = True

experiment_id = train(
    env_class=SearchEnv,
    env_args={
        "search": search,
        "judge_base_url": JUDGE_BASE_URL,
        "judge_api_key": JUDGE_API_KEY,
        "judge_model": JUDGE_MODEL,
    },
    train_dataset=train_data,
    eval_dataset=eval_data,
    prefix=EXPERIMENT_PREFIX,
    api_key=API_KEY,
    base_url=BASE_URL,
    local_modules=[cgft],
    pip_dependencies=["pinecone", "openai"],
    experiment_name=EXPERIMENT_NAME,
    validate_env_remotely=VALIDATE_REMOTELY,
)

# Legacy alternative (also works):
# from cgft.envs.search_env import SearchEnv
# train(env_class=SearchEnv, env_args={...}, ...)

## Monitoring Training: What to Expect

### Early Training Noise

**Early training**: At the start of a training run, rewards will fluctuate and metrics may be noisy. This is completely normal - the model is still learning basic patterns and its outputs are unstable. Give it some time (usually a few dozen steps) and the signal will clean up and you'll start seeing clear trends.

**Exploration before exploitation**: Ideally, you want to see pass@k climbing first before average reward increases significantly. This means your model is exploring the solution space and learning to solve more prompts (breadth) before it optimizes for higher rewards on those prompts (depth). If average reward shoots up while pass@k stays low, you're likely exploiting a narrow set of easy prompts rather than building robust capabilities.

**Healthy training trajectory**: In a well-configured training run, expect pass@k to increase early as the model learns to solve more distinct prompts. Average reward should follow but lag behind. Eventually both should plateau as the model saturates your training distribution.

**Warning signs**:

- pass@k flat while average reward increases → model is overfitting to a narrow subset of prompts
- both metrics stagnate early → training distribution may be too hard, reward signal too sparse